# Speaker Recognition — CNV FINN Accelerator Inference on KV260

Inference for the Speaker Recognition FINN CNV model

**NOT WORKING**: Inference presents data transfer bottlenecks and deadlocks, the accelerator never produces an output. Happening both for 4-bits and 8-bits models.

Check the kria_cnv_inference_OLD.ipynb to see a previous CNV architecture sintetized in 4-bits, that was able to run in the target HW, for a understanding on the behavior of CNV models for Speaker Recognition in HW. Summary of those metrics:

| Metric | Value |
|--------|-------|
| Test (Hardware) | **97.97%** |
| Mean Latency | **8.106 ms** |
| Throughput | **123 samples/s** |

---
## 0. Imports and configuration

In [1]:
import os, sys, json, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings("ignore")

# ── Model selection ───────────────────────────────────────────────────────────
MODEL_WIDTH = "4-bits"          # "4-bits" or "8-bits"

# ── Paths ─────────────────────────────────────────────────────────────────────
HERE                = Path(".")
DRIVER_DIR          = HERE / MODEL_WIDTH
BITFILE             = DRIVER_DIR / "bitfile" / "finn-accel.bit"
DEPLOY_EXTRAS       = DRIVER_DIR / "deploy_extras"
RUNTIME_WEIGHT_DIR  = DRIVER_DIR / "runtime_weights"    # Fix #4: correct path

assert BITFILE.exists(),                 f"Bitfile not found: {BITFILE}"
assert DEPLOY_EXTRAS.exists(),           f"deploy_extras/ not found: {DEPLOY_EXTRAS}"
assert (DRIVER_DIR / "driver.py").exists(), f"driver.py not found in {DRIVER_DIR}"

# ── Load metadata written by the FINN synthesis notebook ──────────────────────
with open(DEPLOY_EXTRAS / "classes.json") as f:
    META = json.load(f)

CLASSES        = META["classes"]
NUM_CLASSES    = META["num_classes"]
USE_CHANNELS   = META["use_channels"]      # "I" or "IQ"
N_MFCC         = META["n_mfcc"]            # 20
MFCC_FRAMES    = META["mfcc_frames"]       # 64
HOP_LENGTH     = META["hop_length"]        # 2250
N_FFT          = META["n_fft"]             # 2048
SAMPLING_FREQ  = META["sampling_freq"]     # 48000
INPUT_CHANNELS = 1 if USE_CHANNELS == "I" else 2
WEIGHT_BITS    = META["weight_bits"]
ACT_BITS       = META["act_bits"]

# ── Quantization range from the model's actual bit width ──────────────
Q_MIN = -(2 ** (ACT_BITS - 1))
Q_MAX =  (2 ** (ACT_BITS - 1)) - 1

# CNV_INPUT_SHAPE is NHWC — the layout the FINN driver expects at the DMA
CNV_INPUT_SHAPE = (1, N_MFCC, MFCC_FRAMES, INPUT_CHANNELS)  # (1, 20, 64, 1)
CNV_INPUT_ELEMS = int(np.prod(CNV_INPUT_SHAPE))            

# ── Load quantization parameters ──────────────────────────────────────────────
scale_arr   = np.load(DEPLOY_EXTRAS / "input_scale.npy")
INPUT_SCALE = float(scale_arr[0])
INPUT_ZP    = float(scale_arr[1])

# ── Real-time budget: one inference per MFCC frame ────────────────────────────
FRAME_DURATION_MS = HOP_LENGTH / SAMPLING_FREQ * 1000

print("── CNV deployment metadata ──────────────────────────────────")
print(f"  Architecture    : CNV W{WEIGHT_BITS}A{ACT_BITS}  (BNN-PYNQ style)")
print(f"  Classes ({NUM_CLASSES})     : {CLASSES}")
print(f"  Channel mode    : {USE_CHANNELS} ({INPUT_CHANNELS} input channel(s))")
print(f"  CNV input shape : {CNV_INPUT_SHAPE}  — (N, H, W, C) NHWC")
print(f"  Input elements  : {CNV_INPUT_ELEMS} per sample")
print(f"  Quant range     : [{Q_MIN}, {Q_MAX}]  ({ACT_BITS}-bit signed)")
print(f"  Input scale     : {INPUT_SCALE:.6f}   zero_point: {INPUT_ZP}")
print(f"  MFCC params     : n_mfcc={N_MFCC}  frames={MFCC_FRAMES}  hop={HOP_LENGTH}  n_fft={N_FFT}")
print(f"  Sampling freq   : {SAMPLING_FREQ} Hz")
print(f"  Real-time budget: {FRAME_DURATION_MS:.1f} ms per MFCC frame")
print("─────────────────────────────────────────────────────────────")

── CNV deployment metadata ──────────────────────────────────
  Architecture    : CNV W4A4  (BNN-PYNQ style)
  Classes (10)     : ['f0001', 'f0002', 'f0003', 'f0004', 'f0005', 'm0001', 'm0002', 'm0003', 'm0004', 'm0005']
  Channel mode    : I (1 input channel(s))
  CNV input shape : (1, 20, 64, 1)  — (N, H, W, C) NHWC
  Input elements  : 1280 per sample
  Quant range     : [-8, 7]  (4-bit signed)
  Input scale     : 0.701747   zero_point: 0.0
  MFCC params     : n_mfcc=20  frames=64  hop=2250  n_fft=2048
  Sampling freq   : 48000 Hz
  Real-time budget: 46.9 ms per MFCC frame
─────────────────────────────────────────────────────────────


---
## 1. Load the FINN accelerator overlay

In [2]:
sys.path.insert(0, str(DRIVER_DIR))

# ── import io_shape_dict directly ─────────────────
from driver import FINNExampleOverlay, io_shape_dict

print(f"Loading bitstream: {BITFILE.name} …")
t0 = time.time()

# ── Explicit batch_size and correct runtime_weight_dir ───────────
runtime_wd = str(RUNTIME_WEIGHT_DIR) if RUNTIME_WEIGHT_DIR.exists() else ""

accel = FINNExampleOverlay(
    bitfile_name       = str(BITFILE),
    platform           = "zynq-iodma",
    io_shape_dict      = io_shape_dict,
    batch_size         = 1,
    runtime_weight_dir = runtime_wd,
)

print(f"Overlay loaded in {time.time()-t0:.1f} s")

# ── Sanity check: print what the driver expects at its interfaces ─────────────
print("\n── Driver I/O shape dictionary ──────────────────────────────")
print(f"  ishape_normal : {io_shape_dict.get('ishape_normal')}")
print(f"  oshape_normal : {io_shape_dict.get('oshape_normal')}")
print(f"  idt           : {io_shape_dict.get('idt')}")
print(f"  odt           : {io_shape_dict.get('odt')}")
print(f"  num_inputs    : {io_shape_dict.get('num_inputs', 1)}")
print(f"  num_outputs   : {io_shape_dict.get('num_outputs', 1)}")
print(f"  runtime_weight_dir : {runtime_wd or '(none, skipping)'}")

# Cache the expected input / output shapes as plain tuples for fast assertions
ISHAPE_NORMAL = tuple(io_shape_dict["ishape_normal"][0])
OSHAPE_NORMAL = tuple(io_shape_dict["oshape_normal"][0])

assert ISHAPE_NORMAL == CNV_INPUT_SHAPE, (
    f"Driver expects {ISHAPE_NORMAL} but notebook computed CNV_INPUT_SHAPE={CNV_INPUT_SHAPE}. "
    f"The deployment and the metadata went out of sync — re-run the synthesis notebook."
)
print(f"\n  ✓ driver ishape_normal matches CNV_INPUT_SHAPE = {CNV_INPUT_SHAPE}")

Loading bitstream: finn-accel.bit …


Overlay loaded in 2.0 s

── Driver I/O shape dictionary ──────────────────────────────
  ishape_normal : [(1, 20, 64, 1)]
  oshape_normal : [(1, 10)]
  idt           : [INT4]
  odt           : [INT16]
  num_inputs    : 1
  num_outputs   : 1
  runtime_weight_dir : (none, skipping)

  ✓ driver ishape_normal matches CNV_INPUT_SHAPE = (1, 20, 64, 1)


---
## 2. Inference helpers (CNV-specific)

All input data must reach the accelerator in **NCHW layout** —
`(1, INPUT_CHANNELS, N_MFCC, MFCC_FRAMES)` = `(1, 1, 20, 64)` for I-channel.

In [3]:
def quantize_input(x_float: np.ndarray,
                   scale: float = INPUT_SCALE,
                   zero_point: float = INPUT_ZP,
                   q_min: int = Q_MIN,
                   q_max: int = Q_MAX) -> np.ndarray:
    """Replicate the Brevitas QuantIdentity (IntKActPerTensorFloat) step.

    Formula: q = clip(round(x / scale) + zero_point, q_min, q_max)
    """
    x_q = np.round(x_float / scale) + zero_point
    return np.clip(x_q, q_min, q_max).astype(np.int8)


def _validate_input_for_dma(arr: np.ndarray) -> None:
    """Pre-DMA sanity checks — cheap, and catch the failure modes that
    otherwise manifest as a silent DMA hang."""
    if arr.dtype != np.int8:
        raise TypeError(f"Input dtype must be int8 (got {arr.dtype}). "
                        f"Do NOT .view(np.uint8); the driver packer needs the "
                        f"signed interpretation of the bytes.")
    if arr.shape != ISHAPE_NORMAL:
        raise ValueError(f"Input shape {arr.shape} != driver ISHAPE_NORMAL {ISHAPE_NORMAL}.")
    if not arr.flags["C_CONTIGUOUS"]:
        raise ValueError("Input must be C-contiguous; wrap with np.ascontiguousarray().")
    # Fix #1: range check against the MODEL's actual bit width, not int8.
    if arr.min() < Q_MIN or arr.max() > Q_MAX:
        raise ValueError(f"Input values [{arr.min()}, {arr.max()}] outside legal "
                         f"range [{Q_MIN}, {Q_MAX}] for {ACT_BITS}-bit signed. "
                         f"The FINN packer will mis-pack these and stall the pipeline.")


def run_single(x_q: np.ndarray) -> int:
    """Run inference on one sample.

    Parameters
    ----------
    x_q : np.ndarray, int8
        NCHW-ordered sample. Shape (1, C, H, W), (C, H, W), or (1, H, W) for C=1.
        Must already be quantized (int8 representation of the signed K-bit value).
    """
    if x_q.dtype != np.int8:
        x_q = x_q.astype(np.int8)

    # Normalize shape to (C, H, W), then transpose to (H, W, C), then add N=1.
    arr  = x_q.reshape(INPUT_CHANNELS, N_MFCC, MFCC_FRAMES)               # (C, H, W)
    nhwc = np.ascontiguousarray(arr.transpose(1, 2, 0)[np.newaxis])       # (1, H, W, C)

    _validate_input_for_dma(nhwc)
    obuf = accel.execute([nhwc])
    out  = obuf[0] if isinstance(obuf, list) else obuf
    return int(np.argmax(out.flatten()[:NUM_CLASSES]))


def run_batch(X_q: np.ndarray, batch_size: int = 1) -> np.ndarray:
    """Run inference on a dataset, one sample per DMA call.
    """
    if batch_size != 1:
        raise NotImplementedError(
            "This overlay was initialized with batch_size=1. "
            "Re-initialize FINNExampleOverlay with a larger batch_size to use batched DMA.")

    if X_q.dtype != np.int8:
        X_q = X_q.astype(np.int8)

    N = X_q.shape[0]
    # Transpose the whole array once: NCHW (N,C,H,W) → NHWC (N,H,W,C).
    X_nhwc = np.ascontiguousarray(X_q.transpose(0, 2, 3, 1))
    preds  = np.zeros(N, dtype=np.int64)

    print(f"Running {N} samples (batch_size={batch_size}) …")
    t0 = time.perf_counter()

    for i in range(N):
        sample = X_nhwc[i : i + 1]           # shape (1, H, W, C), int8, contiguous
        _validate_input_for_dma(sample)
        out = accel.execute([sample])        
        out0 = out[0] if isinstance(out, list) else out
        preds[i] = int(np.argmax(out0.flatten()[:NUM_CLASSES]))
        if i % 50 == 0:
            print(f"  {i+1:>5}/{N}  ({time.perf_counter()-t0:.1f} s)", end="\r")

    elapsed = time.perf_counter() - t0
    print(f"\nDone — {N} samples in {elapsed:.2f} s  ({N/elapsed:.0f} samples/s)")
    return preds


print("Inference helpers defined.")
print(f"  ISHAPE_NORMAL   : {ISHAPE_NORMAL}")
print(f"  OSHAPE_NORMAL   : {OSHAPE_NORMAL}")
print(f"  Legal int range : [{Q_MIN}, {Q_MAX}]  ({ACT_BITS}-bit signed)")

Inference helpers defined.
  ISHAPE_NORMAL   : (1, 20, 64, 1)
  OSHAPE_NORMAL   : (1, 10)
  Legal int range : [-8, 7]  (4-bit signed)


---
## 3. Batch accuracy on the full test set

In [ ]:
print("Loading pre-quantized test set …")
X_test_q = np.load(DEPLOY_EXTRAS / "X_test_q.npy")   # (N, 1, 20, 64) int8
y_test   = np.load(DEPLOY_EXTRAS / "y_test.npy")     # (N,)           int64

print(f"  X_test_q : {X_test_q.shape}  dtype={X_test_q.dtype}")
print(f"  y_test   : {y_test.shape}   dtype={y_test.dtype}")

y_pred   = run_batch(X_test_q, batch_size=1)
accuracy = float(np.mean(y_pred == y_test))

print(f"\nHardware accuracy : {accuracy*100:.2f}%  "
      f"({int(accuracy*len(y_test))}/{len(y_test)} correct)")

Loading pre-quantized test set …
  X_test_q : (493, 1, 20, 64)  dtype=int8
  y_test   : (493,)   dtype=int64
Running 493 samples (batch_size=1) …


---
## 4. Throughput and latency statistics

In [ ]:
N_REPS  = 200
x_bench = np.ascontiguousarray(X_test_q[0:1].transpose(0, 2, 3, 1))   # (1,20,64,1) int8
_validate_input_for_dma(x_bench)

# Warm-up: prime the AXI DMA and PL pipeline
for _ in range(5):
    accel.execute([x_bench])                 

latencies = []
for _ in range(N_REPS):
    t = time.perf_counter()
    accel.execute([x_bench])                 
    latencies.append((time.perf_counter() - t) * 1000)

latencies = np.array(latencies)

print(f"Per-sample latency over {N_REPS} runs")
print(f"  Mean        : {latencies.mean():.3f} ms")
print(f"  Median      : {np.median(latencies):.3f} ms")
print(f"  P99         : {np.percentile(latencies, 99):.3f} ms")
print(f"  Min         : {latencies.min():.3f} ms")
print(f"  Max         : {latencies.max():.3f} ms")
print(f"  Throughput  : {1000/latencies.mean():.0f} samples/s")

---
## 5. Confusion matrix and per-class accuracy

In [ ]:
cm      = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ["d", ".2f"],
    ["Raw counts", "Normalised (recall per class)"]
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax,
                linewidths=0.3, cbar=True)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix — {title}")
    ax.tick_params(axis="x", rotation=45)

plt.suptitle(
    f"CNV W{WEIGHT_BITS}A{ACT_BITS} Hardware Inference  |  "
    f"Accuracy: {accuracy*100:.2f}%  |  Channels: {USE_CHANNELS}",
    y=1.01)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrix.png")
print("\n" + classification_report(y_test, y_pred, target_names=CLASSES))

---
## 6. Summary

In [ ]:
print("╔" + "═"*66 + "╗")
print("║" + " CNV Speaker Recognition — KV260 Deployment Results".center(66) + "║")
print("╚" + "═"*66 + "╝")
print()
print(f"  Bitfile        : {BITFILE}")
print()
print(f"  Test-set accuracy                 : {accuracy*100:.2f}%")
print(f"  Mean latency per sample           : {latencies.mean():.3f} ms")
print(f"  Throughput                        : {1000/latencies.mean():.0f} samples/s")
print("╚" + "═"*66 + "╝")